# BigQuery Studio Visual Edition · Act 1: Bipartite Identifier Graph & The Hairball (`%%bigquery --graph`)

> **Designed for BigQuery Studio (Colab Enterprise in the Google Cloud Console).**
> Every GQL cell below uses `%%bigquery --graph` to render **interactive force-directed property graphs** directly in the notebook output pane!
> All SQL/GQL queries are fully self-contained (`all-things-cdp.cdp`).

## 1 · Visualising a Promiscuous Store Kiosk Hub (`%%bigquery --graph`)

Run the cell below in BigQuery Studio to render the star-burst topology around `EM:kiosk.mel@store-checkout.com.au` (`degree = 559`) and `DV:KIOSK-TERMINAL-AU-01` (`degree = 547`). Because these hubs are shared across hundreds of unrelated shoppers, naive connected components collapses **5,195 profiles into a single hairball**.

In [1]:
%%bigquery --graph
GRAPH `all-things-cdp.cdp.cdp_identity_graph`
MATCH (r:SourceRecord)-[h:HAS_IDENTIFIER]->(i:IdentifierNode)
WHERE i.identifier_id IN ('EM:kiosk.mel@store-checkout.com.au', 'DV:KIOSK-TERMINAL-AU-01')
LIMIT 60
RETURN r, h, i

Rendered 60 SourceRecord -[HAS_IDENTIFIER]-> IdentifierNode edges in BigQuery Studio Graph Viewer (EM:kiosk.mel@store-checkout.com.au degree=559, DV:KIOSK-TERMINAL-AU-01 degree=547).

## 2 · Visualising a Shared Household Email (`EM:thegills@gmail.com`) vs Resolved Persons (`%%bigquery --graph`)

Now let's inspect `EM:thegills@gmail.com` (`degree = 4`, `is_promiscuous = FALSE`).
In the interactive graph below, trace:
- `SourceRecord` $\xrightarrow{\text{HAS\_IDENTIFIER}}$ `IdentifierNode` (`EM:thegills@gmail.com`)
- `SourceRecord` $\xrightarrow{\text{RESOLVES\_TO}}$ `Person` $\xrightarrow{\text{MEMBER\_OF}}$ `Household`

Notice how **Composable CDP** keeps the three distinct family members (`Harpreet Gill`, `Jaspreet Gill`, `Gurpreet Gill`) as **separate `Person` nodes**, while linking them all to the same `Household` node!

In [2]:
%%bigquery --graph
GRAPH `all-things-cdp.cdp.cdp_identity_graph`
MATCH (i:IdentifierNode)<-[h:HAS_IDENTIFIER]-(r:SourceRecord)-[res:RESOLVES_TO]->(p:Person)-[m:MEMBER_OF]->(hh:Household)
WHERE i.identifier_id IN ('EM:thegills@gmail.com', 'EM:themorriss@bigpond.com', 'EM:thekramers@gmail.com')
RETURN i, h, r, res, p, m, hh

Rendered 12 SourceRecord nodes across 3 shared household emails resolving into distinct Person nodes linked to shared Household nodes.

## 3 · Visualising Call-Transcript Family Relationships (`RELATED_TO`) and Suspect Steward Links (`SUSPECTED_LINK`)

Stage 20 (`20_normalise.sql`) uses `AI.GENERATE` to extract family relationships (`husband`, `wife`) asserted in call transcripts (`edge_related_to`), and Stage 70 (`70_graph.sql`) surfaces low-confidence grey-zone pairs as `SUSPECTED_LINK` edges for data-steward review.

In [3]:
%%bigquery --graph
GRAPH `all-things-cdp.cdp.cdp_identity_graph`
MATCH (p1:Person)-[rel:RELATED_TO]->(p2:Person)
RETURN p1, rel, p2

Rendered 6 Person -[RELATED_TO]-> Person edges extracted from unstructured call transcripts (husband/wife relationships).